# One-dimensional conservative deposition

The complete one-dimensional pipeline: hydro grid, initialized rays, JAX/Diffrax tracing, segment fields, a line deposition mesh, and exact conservative deposition.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from pyGATH.fields import (
    build_linear_deposition_mesh_from_grid,
    deposit_simplicial_power_to_mesh,
    simplicialise_sheet_fields,
)
from pyGATH.io import load_simulation_config

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
simulation = load_simulation_config(
    root / "configs/example_configs/uniform_1d_deposition.toml"
)

In [ ]:
with simulation.reporting():
    grid = simulation.build_grid()
    beams = simulation.load_beams()
    initial_rays = simulation.initialize_rays(grid, beams=beams)
    trace = simulation.trace_rays(initial_rays, grid)
    source = simplicialise_sheet_fields(
        trace.sheet_fields, dimension=1, fields="inverse_brems_deposition"
    )
    target = build_linear_deposition_mesh_from_grid(grid)
    deposition = deposit_simplicial_power_to_mesh(source, target)
print(f"source={deposition.source_power:.6e} W")
print(f"deposited={deposition.deposited_power:.6e} W")
print(f"conservation error={deposition.conservation_error:.3e} W")

In [ ]:
centres_um = np.asarray(target.cell_centres[:, 0]) * 1e6
plt.plot(centres_um, np.asarray(deposition.power_density))
plt.xlabel("x [um]")
plt.ylabel("deposited power density [W/m^3]")
plt.grid(alpha=0.25)